In [1]:
import torch
import torch.nn as nn
import random , math , time
SEED  = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda")

In [2]:
from datasets import load_dataset
dataset = load_dataset("bentrevett/multi30k")
train_dataset = dataset["train"]
val_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [3]:
print(train_dataset[0])

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}


In [4]:
len(train_dataset),len(val_dataset),len(test_dataset)

(29000, 1014, 1000)

In [5]:
import spacy 
tokenizer ={}
tokenizer["en"] = spacy.load("en_core_web_sm",disable= ["tagger", "parser", "ner", "lemmatizer", "attribute_ruler"])
tokenizer["de"] = spacy.load("de_core_news_sm",disable=["tagger", "parser", "ner", "lemmatizer", "attribute_ruler"])

In [6]:
def tokenization(text,language):
    dic = tokenizer[language](text)
    return [token.text for token in dic]
tokenization(train_dataset["en"][0],"en")

['Two',
 'young',
 ',',
 'White',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.']

In [7]:
tokenization(train_dataset["de"][0],"de")

['Zwei',
 'junge',
 'weiße',
 'Männer',
 'sind',
 'im',
 'Freien',
 'in',
 'der',
 'Nähe',
 'vieler',
 'Büsche',
 '.']

In [8]:
from collections import Counter
word_counter = {"en": Counter(),"de":Counter()}
for ln in ["en","de"]:
    for doc in tokenizer[ln].pipe(train_dataset[ln],batch_size = 2000):
        tokens = [token.text.lower() for token in doc]
        word_counter[ln].update(tokens)
min_freq = 2
special_token = ["<unk>","<pad>","<sos>","<eos>"]
word2index = {"en":{},"de":{}}
index2word = {"en":{},"de":{}}

for ln in ["en","de"]:
    word2index[ln] = {word : idx for idx, word in enumerate(special_token)}
for ln in ["en","de"]:
    for word , count in word_counter[ln].items():
        if count >= min_freq:
            word2index[ln][word] = len(word2index[ln])
for ln in ["en", "de"]:
    index2word[ln] = {idx: word for word, idx in word2index[ln].items()}

In [9]:
len(word2index["de"]),len(word2index["en"])

(7853, 5893)

In [10]:
index2word["en"][0]

'<unk>'

In [11]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

unk_idx, pad_idx, sos_idx, eos_idx = 0, 1, 2, 3

# Helper function to convert tokenized text to token indices
def text_transform_fn(tokenize_text, ln):
    # Retrieve indices from word2index, lowercasing tokens to match vocabulary keys
    return [word2index[ln].get(token.lower(), unk_idx) for token in tokenize_text]

def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

# Add SOS and EOS tokens, and convert list of indices to a PyTorch tensor
def tensor_transform(token_ids):
    return torch.cat([
        torch.tensor([sos_idx], dtype=torch.long),
        torch.tensor(token_ids, dtype=torch.long),
        torch.tensor([eos_idx], dtype=torch.long)
    ])

# Helper function to construct sequential transforms for each language,
# avoiding late-binding issues in loops
def build_transform(ln):
    return sequential_transforms(
        lambda txt: tokenization(txt, ln),
        lambda tokens: text_transform_fn(tokens, ln),
        tensor_transform
    )

text_transform = {}
for ln in ["en", "de"]:
    text_transform[ln] = build_transform(ln)

# Collate function to pad batch sequences
def collate_fn(batch):
    de_batch, en_batch = [], []
    for sample in batch:
        de_batch.append(text_transform["de"](sample["de"]))
        en_batch.append(text_transform["en"](sample["en"]))
    
    de_batch = pad_sequence(de_batch, padding_value=pad_idx, batch_first=True)
    en_batch = pad_sequence(en_batch, padding_value=pad_idx, batch_first=True)
    return de_batch, en_batch


In [12]:
# Example usage of the collate_fn with PyTorch DataLoader
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset,batch_size= 500,shuffle=False,collate_fn=collate_fn)
test_loader = DataLoader(test_dataset,batch_size=500,shuffle=False,collate_fn=collate_fn)

# Retrieve a single batch to verify
src_batch, tgt_batch = next(iter(train_loader))
print(f"Source batch shape (de): {src_batch.shape}")
print(f"Target batch shape (en): {tgt_batch.shape}")


Source batch shape (de): torch.Size([128, 25])
Target batch shape (en): torch.Size([128, 26])


In [13]:
src_batch[0],tgt_batch[0]

(tensor([   2,   21, 3826,  829,  197,  139,  554,   49,   50,  395,  194,   11,
           12,  648,  566,   16,    3,    1,    1,    1,    1,    1,    1,    1,
            1]),
 tensor([   2,   21, 3448, 3763,   51,  557,   48, 1736,   14,    3,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1]))

In [14]:
len(train_loader),len(val_loader),len(test_loader)

(227, 3, 2)

In [15]:
src_batch.shape[0]

128

In [16]:
class seq2seq(nn.Module):
    def __init__(self, encoder, decoder, pad_idx, device):
        super(seq2seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.device = device
    
    def create_mask(self, src):
        mask = (src == self.pad_idx) # Shape: [batch_size, src_len]
        return mask

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_length = trg.shape[1]
        src_length = src.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_length, trg_vocab_size).to(self.device)
        attentions = torch.zeros(batch_size, trg_length, src_length).to(self.device)
        
        # Encoder forward pass
        encoder_outputs, hidden = self.encoder(src)
        
        # First input to the decoder is the <sos> tokens
        input_ = trg[:, 0]
        mask = self.create_mask(src)
        
        for t in range(1, trg_length):
            # Decoder forward pass
            output, hidden, attention = self.decoder(input_, hidden, encoder_outputs, mask)
            
            # Store outputs and attentions
            outputs[:, t, :] = output
            attentions[:, t, :] = attention
            
            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_ = trg[:, t] if teacher_force else top1
            
        return outputs, attentions


In [17]:
class encoder(nn.Module):
    def __init__(self, input_dim, embedd_dim, hid_dim, dropout):
        super(encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, embedd_dim)
        self.rnn = nn.GRU(embedd_dim, hid_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)
        
    def forward(self, src):
        # src shape: [batch_size, src_len]
        embedded = self.dropout(self.embedding(src)) # [batch_size, src_len, embedd_dim]
        
        # Calculate sequence lengths dynamically by counting non-padding tokens (pad_idx is 1)
        src_lengths = (src != 1).sum(dim=1).to("cpu")
        
        # Pack sequence
        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded, src_lengths, batch_first=True, enforce_sorted=False
        )
        
        packed_outputs, hidden = self.rnn(packed_embedded)
        
        # Unpack sequence
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        
        # Concat bidirectional hidden states: hidden shape is [2, batch_size, hid_dim]
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)))
        
        return outputs, hidden


In [18]:
class attention(nn.Module):
    def __init__(self, hid_dim):
        super(attention, self).__init__()
        self.v = nn.Linear(hid_dim, 1)
        self.w = nn.Linear(hid_dim, hid_dim)
        self.u = nn.Linear(hid_dim * 2, hid_dim)
        
    def forward(self, hidden, encoder_outputs, mask):
        # hidden shape: [batch_size, hid_dim]
        # encoder_outputs shape: [batch_size, src_len, hid_dim * 2]
        # mask shape: [batch_size, src_len]
        
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        # Repeat decoder hidden state src_len times
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1) # [batch_size, src_len, hid_dim]
        
        # energy = tanh(W * s + U * h)
        energy = torch.tanh(self.w(hidden) + self.u(encoder_outputs)) # [batch_size, src_len, hid_dim]
        
        # attention = V * energy
        attention = self.v(energy).squeeze(2) # [batch_size, src_len]
        
        # Mask out padding tokens
        if mask is not None:
            attention = attention.masked_fill(mask, -1e10)
            
        return torch.softmax(attention, dim=1)


In [19]:
class decoder(nn.Module):
    def __init__(self, output_dim, embedd_dim, hid_dim, dropout, attention):
        super(decoder, self).__init__()
        self.output_dim = output_dim
        self.attention = attention
        
        self.embedding = nn.Embedding(output_dim, embedd_dim)
        self.rnn = nn.GRU(embedd_dim + hid_dim * 2, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim + hid_dim * 2 + embedd_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_, hidden, encoder_outputs, mask):
        # input_: [batch_size]
        # hidden: [batch_size, hid_dim]
        # encoder_outputs: [batch_size, src_len, hid_dim * 2]
        # mask: [batch_size, src_len]
        
        input_ = input_.unsqueeze(1) # [batch_size, 1]
        embedded = self.dropout(self.embedding(input_)) # [batch_size, 1, embedd_dim]
        
        # Calculate attention weights
        a = self.attention(hidden, encoder_outputs, mask) # [batch_size, src_len]
        a = a.unsqueeze(1) # [batch_size, 1, src_len]
        
        # context = a * encoder_outputs
        context = torch.bmm(a, encoder_outputs) # [batch_size, 1, hid_dim * 2]
        
        # GRU input: concat embedded input and context
        rnn_input = torch.cat((embedded, context), dim=2) # [batch_size, 1, embedd_dim + hid_dim * 2]
        
        # Pass through GRU
        output, hidden_next = self.rnn(rnn_input, hidden.unsqueeze(0))
        
        # Squeeze outputs for classifier projection
        output = output.squeeze(1) # [batch_size, hid_dim]
        context = context.squeeze(1) # [batch_size, hid_dim * 2]
        embedded = embedded.squeeze(1) # [batch_size, embedd_dim]
        
        prediction = self.fc(torch.cat((output, context, embedded), dim=1)) # [batch_size, output_dim]
        hidden_next = hidden_next.squeeze(0) # [batch_size, hid_dim]
        
        return prediction, hidden_next, a.squeeze(1)


In [20]:
def train_epoch(model, iterator, optimizer, criterion, teacher_forcing_ratio=0.5, clip=1.0):
    model.train()
    epoch_loss = 0
    
    for src, trg in iterator:
        src = src.to(device)
        trg = trg.to(device)
        
        optimizer.zero_grad()
        
        # Pass teacher_forcing_ratio dynamically
        outputs, _ = model(src, trg, teacher_forcing_ratio=teacher_forcing_ratio)
        
        output_dim = outputs.shape[-1]
        
        # Squeeze sequence elements to compute CrossEntropy over flat dimension
        # Exclude <sos> token at index 0 from output and target
        outputs = outputs[:, 1:, :].contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)
        
        loss = criterion(outputs, trg)
        loss.backward()
        
        # Clip gradient norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in iterator:
            src = src.to(device)
            trg = trg.to(device)
            
            # Disable teacher forcing during evaluation
            outputs, _ = model(src, trg, teacher_forcing_ratio=0.0)
            
            output_dim = outputs.shape[-1]
            outputs = outputs[:, 1:, :].contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)
            
            loss = criterion(outputs, trg)
            epoch_loss += loss.item()
            
    return epoch_loss / len(iterator)

def translate_sentence(sentence, model, device, max_len=50):
    model.eval()
    
    # Run the source through sequential transforms
    src_tensor = text_transform["de"](sentence).unsqueeze(0).to(device)
    
    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)
        
    mask = model.create_mask(src_tensor)
    
    trg_indexes = [sos_idx]
    
    # Autoregressively generate English target sentence
    for i in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long, device=device)
        
        with torch.no_grad():
            output, hidden, _ = model.decoder(trg_tensor, hidden, encoder_outputs, mask)
            
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        
        if pred_token == eos_idx:
            break
            
    # Map token indices to words
    trg_tokens = [index2word["en"][idx] for idx in trg_indexes]
    
    # Strip special tokens and format translation
    return ' '.join(t for t in trg_tokens if t not in ["<sos>", "<eos>", "<pad>"])


In [21]:
import torch.optim as optim

input_dim = len(word2index["de"])
output_dim = len(word2index["en"])
embed_dim = 128
hid_dim = 256
dropout = 0.5

# Instantiate components and Seq2Seq model
enc = encoder(input_dim, embed_dim, hid_dim, dropout).to(device)
attn = attention(hid_dim).to(device)
dec = decoder(output_dim, embed_dim, hid_dim, dropout, attn).to(device)
model = seq2seq(enc, dec, pad_idx, device).to(device)

# Loss function (ignoring padding) and optimizer with L2 regularization
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)


In [24]:
import math

N_EPOCHS = 30
best_valid_loss = float('inf')

print("Starting training...")

for epoch in range(N_EPOCHS):
    # Scheduled Teacher Forcing Decay (from 0.5 down to ~0.3 over 10 epochs)
    teacher_forcing_ratio = max(0.1, 0.5 - (epoch * 0.02))
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, teacher_forcing_ratio=teacher_forcing_ratio)
    val_loss = evaluate(model, val_loader, criterion)
    
    train_ppl = math.exp(train_loss) if train_loss < 20 else float('inf')
    val_ppl = math.exp(val_loss) if val_loss < 20 else float('inf')
    
    print(f'Epoch: {epoch+1:02} | Teacher Forcing: {teacher_forcing_ratio:.2f}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {train_ppl:.3f}')
    print(f'\t Val. Loss: {val_loss:.3f} |  Val. PPL: {val_ppl:.3f}')
    
    # Save the best model checkpoints based on validation loss (Early Stopping)
    if val_loss < best_valid_loss:
        best_valid_loss = val_loss
        torch.save(model.state_dict(), 'best-model.pt')
        print(f'\t>> Saved best model checkpoint (Val Loss: {val_loss:.3f})')

# Load the best checkpoint before evaluating on test dataset
print("\nLoading best model checkpoint for final testing...")
model.load_state_dict(torch.load('best-model.pt', weights_only=True))

# Final Test set evaluation
test_loss = evaluate(model, test_loader, criterion)
test_ppl = math.exp(test_loss) if test_loss < 20 else float('inf')
print(f'Test Loss: {test_loss:.3f} | Test PPL: {test_ppl:.3f}')

# Translate a few sample validation/test sentences to qualitatively demonstrate translation quality
print("\n--- Qualitative Translation Examples ---")
examples = [
    "Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.",
    "Ein Mann in einem blauen Hemd steht auf einer Straße.",
    "Eine Gruppe von Menschen steht vor einem Gebäude."
]
for sentence in examples:
    translation = translate_sentence(sentence, model, device)
    print(f"Source German: {sentence}")
    print(f"Translated English: {translation}\n")


Starting training...
Epoch: 01 | Teacher Forcing: 0.50
	Train Loss: 3.477 | Train PPL: 32.365
	 Val. Loss: 3.498 |  Val. PPL: 33.047
	>> Saved best model checkpoint (Val Loss: 3.498)
Epoch: 02 | Teacher Forcing: 0.48
	Train Loss: 3.109 | Train PPL: 22.395
	 Val. Loss: 3.430 |  Val. PPL: 30.872
	>> Saved best model checkpoint (Val Loss: 3.430)
Epoch: 03 | Teacher Forcing: 0.46
	Train Loss: 2.894 | Train PPL: 18.070
	 Val. Loss: 3.376 |  Val. PPL: 29.251
	>> Saved best model checkpoint (Val Loss: 3.376)
Epoch: 04 | Teacher Forcing: 0.44
	Train Loss: 2.734 | Train PPL: 15.396
	 Val. Loss: 3.329 |  Val. PPL: 27.915
	>> Saved best model checkpoint (Val Loss: 3.329)
Epoch: 05 | Teacher Forcing: 0.42
	Train Loss: 2.626 | Train PPL: 13.819
	 Val. Loss: 3.157 |  Val. PPL: 23.511
	>> Saved best model checkpoint (Val Loss: 3.157)
Epoch: 06 | Teacher Forcing: 0.40
	Train Loss: 2.534 | Train PPL: 12.609
	 Val. Loss: 3.210 |  Val. PPL: 24.775
Epoch: 07 | Teacher Forcing: 0.38
	Train Loss: 2.446 | Tr